In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. 파일 / 시트 불러오기
# =========================
file_path = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
sheet_name = "job_posting_processed"

df = pd.read_excel(file_path, sheet_name=sheet_name)

# =========================
# 2. 카테고리 키워드 사전
# =========================
CATEGORY_KEYWORDS = {
    "프론트엔드": {
        "HTML", "CSS", "JavaScript", "TypeScript",
        "React", "Vue.js", "jQuery", "Ajax", "Bootstrap",
        "Next.js", "SCSS", "WebGL", "Nexacro", "Websquare", "UI/UX"
    },
    "백엔드": {
        "Java", "JSP", "Spring", "Spring Boot", "Spring Framework",
        "Node.js", "PHP", "MySQL", "Oracle", "MSSQL", "MariaDB",
        "PostgreSQL", "SQL", "DB", "DBMS", "RDBMS", "NoSQL", "MongoDB",
        "Django", "Flask", "FastAPI", "NestJS",
        "JPA", "MyBatis", "IBatis", "Servlet",
        "Apache", "WAS", "REST API", "API",
        "전자정부표준프레임워크", ".NET", "ASP", "ASP.NET", "C#", "C++", "C"
    },
    "인공지능/데이터": {
        "Python", "AI", "머신러닝", "딥러닝",
        "TensorFlow", "PyTorch", "OpenCV", "Pandas", "Numpy",
        "Scikit-Learn", "Keras", "JAX", "R",
        "RNN", "LSTM", "CNN", "Transformer",
        "LLM", "LLM API", "GPT", "ChatGPT", "Gemini",
        "Hugging Face", "Hadoop", "AI Agent"
    },
    "인프라/보안": {
        "AWS", "Linux", "Docker", "Kubernetes", "Jenkins",
        "Network", "Kafka", "Unix", "Azure", "클라우드"
    },
    "모바일": {
        "Android", "Android Studio", "Kotlin",
        "Flutter", "React Native", "iOS", "Swift"
    }
}

# =========================
# 3. 전처리 함수
# =========================
def parse_stack(text):
    if pd.isna(text) or str(text).strip() == "":
        return []
    return [token.strip() for token in str(text).split(",") if token.strip()]

# =========================
# 4. 보정 규칙 함수
#    - 애매한 기술스택 처리
# =========================
def apply_context_rules(tokens, scores):
    token_set = set(tokens)

    # Java: 보통 백엔드 기본값
    if "Java" in token_set:
        scores["백엔드"] += 1

    # Python: AI/데이터 기본값, 단 Flask/Django/FastAPI와 같이 있으면 백엔드도 가점
    if "Python" in token_set:
        scores["인공지능/데이터"] += 1
        if {"Django", "Flask", "FastAPI"} & token_set:
            scores["백엔드"] += 2

    # Kotlin: Android/iOS/Flutter/React Native와 같이 있으면 모바일 우선
    if "Kotlin" in token_set:
        if {"Android", "Android Studio", "Flutter", "React Native", "iOS", "Swift"} & token_set:
            scores["모바일"] += 2
        else:
            scores["백엔드"] += 1

    # SQL 계열은 백엔드 쪽으로 조금 더 기울임
    if {"SQL", "MySQL", "Oracle", "MSSQL", "MariaDB", "PostgreSQL"} & token_set:
        scores["백엔드"] += 1

    # AWS + Docker/K8s/Linux 조합이면 인프라 강하게
    if "AWS" in token_set and {"Docker", "Kubernetes", "Linux"} & token_set:
        scores["인프라/보안"] += 2

    # React Native / Flutter / Android / iOS 있으면 모바일 강하게
    if {"React Native", "Flutter", "Android", "iOS"} & token_set:
        scores["모바일"] += 2

    return scores

# =========================
# 5. 공고별 분류 함수
# =========================
def classify_posting(stack_text):
    tokens = parse_stack(stack_text)

    scores = {
        "프론트엔드": 0,
        "백엔드": 0,
        "인공지능/데이터": 0,
        "인프라/보안": 0,
        "모바일": 0
    }

    matched = {
        "프론트엔드": [],
        "백엔드": [],
        "인공지능/데이터": [],
        "인프라/보안": [],
        "모바일": []
    }

    # 기본 매핑 점수
    for token in tokens:
        for category, keyword_set in CATEGORY_KEYWORDS.items():
            if token in keyword_set:
                scores[category] += 1
                matched[category].append(token)

    # 문맥 기반 보정
    scores = apply_context_rules(tokens, scores)

    max_score = max(scores.values())

    if max_score == 0:
        primary = "미분류"
        multi = ""
    else:
        top_categories = [cat for cat, score in scores.items() if score == max_score]
        primary = top_categories[0]   # 동점이면 첫 번째 카테고리
        multi = ", ".join([cat for cat, score in scores.items() if score > 0])

    return pd.Series({
        "PARSED_STACK": ", ".join(tokens),
        "CATEGORY_SCORE_FRONTEND": scores["프론트엔드"],
        "CATEGORY_SCORE_BACKEND": scores["백엔드"],
        "CATEGORY_SCORE_AI_DATA": scores["인공지능/데이터"],
        "CATEGORY_SCORE_INFRA_SECURITY": scores["인프라/보안"],
        "CATEGORY_SCORE_MOBILE": scores["모바일"],
        "PRIMARY_CATEGORY": primary,
        "MULTI_CATEGORIES": multi,
        "MATCHED_FRONTEND": ", ".join(matched["프론트엔드"]),
        "MATCHED_BACKEND": ", ".join(matched["백엔드"]),
        "MATCHED_AI_DATA": ", ".join(matched["인공지능/데이터"]),
        "MATCHED_INFRA_SECURITY": ", ".join(matched["인프라/보안"]),
        "MATCHED_MOBILE": ", ".join(matched["모바일"]),
    })

# =========================
# 6. 적용
# =========================
result_cols = df["TECH_STACK_NORMALIZED"].apply(classify_posting)
df_result = pd.concat([df, result_cols], axis=1)

# 확인
print(df_result[
    [
        "POSTING_ID",
        "COMPANY_NAME",
        "JOB_POSITION",
        "TECH_STACK_NORMALIZED",
        "PRIMARY_CATEGORY",
        "MULTI_CATEGORIES"
    ]
].head(20))

# =========================
# 7. 저장
# =========================
output_path = "공고원본_POSTING_ID포함_정리결과_카테고리분류.xlsx"
df_result.to_excel(output_path, index=False)

print(f"\n저장 완료: {output_path}")

    POSTING_ID COMPANY_NAME      JOB_POSITION  \
0            1          타이아         응용S/W 개발자   
1            2          타이아          웹, 앱 개발자   
2            3          타이아              시스템팀   
3            4    ㈜이글루코퍼레이션             프론트엔드   
4            5      ㈜에프앤가이드               개발자   
5            6        ㈜지니캐디            웹프로그래머   
6            7       금호타이어㈜            완제품 평가   
7            8      (주)세이프디          인공지능 개발자   
8            9     (주)온유소프트            웹 퍼블리셔   
9           10     (주)아이티앱스                기획   
10          11        (주)팜팜               백엔드   
11          12      다온플레이스㈜  IT 및 홍보 마케팅 사업관리   
12          13       (주) 이젠               백엔드   
13          14    (주)플랜비커머스             프로그래머   
14          15       (주)씽크풀         증권 콘텐츠 개발   
15          16       서울아산병원           인공지능 연구   
16          17       ㈜정도UIT               개발자   
17          18     (주)유아이엠디               개발자   
18          19      (주)이즈파크        프론트엔드, 백엔드   
19          20    함께

In [2]:
(df_result["PRIMARY_CATEGORY"] == "미분류").sum()

np.int64(55)